## Notebook — Sync Supabase → Fabric (questionnaire SPV / projets)

### Objectif
Répliquer dans le Lakehouse Fabric (`lakehouse_gold`) les tables liées au **questionnaire** et au **périmètre SPV**, pour les exploiter ensuite dans Power BI / notebooks d'analyse :

| Source Supabase                  | Cible Lakehouse (Delta)               | Volume approx. |
|----------------------------------|---------------------------------------|----------------|
| `project_questionnaire`          | `spv_questionnaire_answers`           | ~ 940 lignes   |
| `questionnaire_field_definitions`| `questionnaire_field_definitions`     | ~ 100 lignes   |
| `spv_affaires`                   | `spv_affaires`                        | ~ 40 lignes    |

### Direction
Supabase (source) → Fabric Lakehouse (cible).
**Mode snapshot complet** : chaque exécution remplace les tables Delta (`overwrite` + `overwriteSchema=true`). Volumétrie trop faible pour justifier de l'incrémental.

### Pré-requis Fabric
1. **Lakehouse `lakehouse_gold` attaché** au notebook (panneau gauche → Add Lakehouse → Existing → lakehouse_gold). Sinon le `saveAsTable` échouera.
2. **Variable library `env-temp`** déjà configurée (cf. `nb_divalto_mouvements_all_sync`) avec au minimum :
   - `SUPABASE_URL`
   - `SUPABASE_PUBLISHABLE_KEY` (clé anon)

### Planification
À brancher dans un pipeline Fabric (Daily / Hourly). Aucune dépendance à d'autres notebooks.

In [ ]:
# ─── 0) Config ──────────────────────────────────────────────────
# Credentials : injectées par le pipeline (Base parameters) si présentes,
# sinon récupérées depuis la variableLibrary (mode exécution manuelle).
try:
    SUPABASE_URL  # mode pipeline ?
except NameError:
    env = notebookutils.variableLibrary.getLibrary("env-temp")
    SUPABASE_URL      = env.SUPABASE_URL
    SUPABASE_ANON_KEY = env.SUPABASE_PUBLISHABLE_KEY

import requests
import pandas as pd
from pyspark.sql import functions as F

GOLD_DB   = "lakehouse_gold"
PAGE_SIZE = 1000   # PostgREST default max

print("[0] Config OK —", SUPABASE_URL)

In [ ]:
# ─── 1) Helper REST paginé (PostgREST) ─────────────────────────────────
def supabase_select(table: str, select: str = "*", filters: dict | None = None):
    """Lecture paginée d'une table Supabase via PostgREST.

    - `table`   : nom de la table
    - `select`  : projection PostgREST (ex. 'id,project_id,valeur')
    - `filters` : dict optionnel {col: 'op.val'} (ex. {'updated_at': 'gt.2026-01-01'})

    Retourne une liste de dict (peut être vide).
    """
    out, offset = [], 0
    base = f"{SUPABASE_URL.rstrip('/')}/rest/v1/{table}"
    params = [("select", select)]
    for k, v in (filters or {}).items():
        params.append((k, v))
    while True:
        h = {
            "apikey":        SUPABASE_ANON_KEY,
            "Authorization": f"Bearer {SUPABASE_ANON_KEY}",
            "Range-Unit":    "items",
            "Range":         f"{offset}-{offset + PAGE_SIZE - 1}",
        }
        r = requests.get(base, headers=h, params=params, timeout=60)
        r.raise_for_status()
        chunk = r.json()
        out.extend(chunk)
        if len(chunk) < PAGE_SIZE:
            break
        offset += PAGE_SIZE
    return out

print("[1] Helper REST prêt")

In [ ]:
# ─── 2) Sync générique : Supabase → Delta (overwrite + schema evolution) ────
TS_COLS_DEFAULT = ("created_at", "updated_at")

def sync_table(src: str, dst: str, select: str = "*", ts_cols=TS_COLS_DEFAULT) -> int:
    rows = supabase_select(src, select)
    print(f"  ← {src:<40s} : {len(rows):>6d} lignes lues")
    if not rows:
        print(f"  ⚠️  Aucune donnée — {GOLD_DB}.{dst} laissée telle quelle.")
        return 0
    pdf = pd.DataFrame(rows)
    # Les colonnes ARRAY de PostgREST arrivent en list[str] : Spark les accepte
    # nativement comme array<string>. Les JSONB arrivent en dict / list : on les
    # sérialise en string pour rester compatible avec Delta managed.
    for col in pdf.columns:
        if pdf[col].map(lambda x: isinstance(x, (dict,))).any():
            import json
            pdf[col] = pdf[col].map(lambda v: json.dumps(v) if isinstance(v, (dict, list)) else v)
    sdf = spark.createDataFrame(pdf)
    # Cast des timestamps ISO -> TimestampType
    for c in ts_cols:
        if c in sdf.columns:
            sdf = sdf.withColumn(c, F.to_timestamp(c))
    n = sdf.count()
    (sdf.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_DB}.{dst}"))
    print(f"  → {GOLD_DB}.{dst:<40s} : {n:>6d} lignes écrites")
    return n

print("[2] Fonction sync_table prête")

In [ ]:
# ─── 3) Exécution ─────────────────────────────────────────────────
print("=" * 70)
print("  Sync Supabase → Fabric : questionnaire SPV")
print("=" * 70)

total = 0
total += sync_table("project_questionnaire",           "spv_questionnaire_answers")
total += sync_table("questionnaire_field_definitions", "questionnaire_field_definitions")
total += sync_table("spv_affaires",                    "spv_affaires")

print("=" * 70)
print(f"  Terminé — {total} lignes au total")
print("=" * 70)

In [ ]:
# ─── 4) (Optionnel) Vue dénormalisée prête à l'usage BI ──────────────────
# Joint les réponses avec leur définition de champ pour avoir le label "propre",
# le type de champ et la note dans une seule table Power BI-friendly.
df_ans  = spark.table(f"{GOLD_DB}.spv_questionnaire_answers")
df_def  = spark.table(f"{GOLD_DB}.questionnaire_field_definitions")

df_enriched = (
    df_ans.alias("a")
          .join(
              df_def.select(
                  F.col("champ_id").alias("def_champ_id"),
                  F.col("label").alias("champ_label"),
                  F.col("type_champ").alias("def_type_champ"),
                  F.col("has_evaluation_risque"),
                  F.col("required"),
                  F.col("order_index"),
              ).alias("d"),
              F.col("a.champ_id") == F.col("d.def_champ_id"),
              "left",
          )
          .drop("def_champ_id")
)

n = df_enriched.count()
(df_enriched.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.spv_questionnaire_enriched"))
print(f"  → {GOLD_DB}.spv_questionnaire_enriched : {n} lignes")

### Notes

- **Sécurité** : la clé publishable (anon) suffit ici car les tables `project_questionnaire`, `questionnaire_field_definitions` et `spv_affaires` ont des policies RLS qui autorisent la lecture authentifiée. Si tu durcis les RLS, ajoute `SUPABASE_SERVICE_ROLE_KEY` dans `env-temp` et utilise-la à la place.
- **Évolution du schéma** : `overwriteSchema=true` absorbe l'ajout/suppression de colonnes côté Supabase sans casse. Les types changeants (ex. `text` → `jsonb`) demandent un drop manuel de la table Delta.
- **Incrémental** : si la volumétrie dépasse ~50k lignes, passer en MERGE sur `updated_at` (PostgREST : `filters={"updated_at": f"gt.{last_run_iso}"}`).
- **Vérification post-run** (à lancer dans un notebook ou SQL endpoint du Lakehouse) :
  ```sql
  SELECT 'answers'     AS t, COUNT(*) FROM lakehouse_gold.spv_questionnaire_answers
  UNION ALL SELECT 'defs',     COUNT(*) FROM lakehouse_gold.questionnaire_field_definitions
  UNION ALL SELECT 'affaires', COUNT(*) FROM lakehouse_gold.spv_affaires
  UNION ALL SELECT 'enriched', COUNT(*) FROM lakehouse_gold.spv_questionnaire_enriched;
  ```